In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

df = pd.read_parquet("../data/processed/churn_clean.parquet")

X = df.drop(columns=["churn", "churn_flag", "customerid"])
y = df["churn_flag"]

num_cols = ["tenure", "monthlycharges", "totalcharges"]
cat_cols = [c for c in X.columns if c not in num_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ]
)

clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=5000, solver="lbfgs"))
])

clf.fit(X_train, y_train)

all_proba = clf.predict_proba(X)[:, 1]

out = pd.DataFrame({
    "customerid": df["customerid"],
    "contract": df["contract"],
    "internetservice": df["internetservice"],
    "tenure": df["tenure"],
    "monthlycharges": df["monthlycharges"],
    "totalcharges": df["totalcharges"],
    "churn_proba": all_proba
})

remaining_map = {
    "Month-to-month": 3,
    "One year": 6,
    "Two year": 12
}

out["remaining_months_assumed"] = out["contract"].map(remaining_map).fillna(3)

out["expected_revenue_loss"] = (
    out["churn_proba"]
    * out["monthlycharges"]
    * out["remaining_months_assumed"]
)

out["risk_segment"] = pd.qcut(out["churn_proba"], q=4, labels=["low", "medium", "high", "very_high"])

out.sort_values("expected_revenue_loss", ascending=False).head(10)


,customerid,contract,internetservice,tenure,monthlycharges,totalcharges,churn_proba,remaining_months_assumed,expected_revenue_loss,risk_segment
3667,7826-VVKWT,Two year,Fiber optic,24,96.55,2263.45,0.297003,12,344.107470,high
336,6680-NENYN,Two year,Fiber optic,43,104.60,4759.85,0.264518,12,332.022518,high
3837,3932-CMDTD,One year,Fiber optic,4,105.65,443.90,0.476871,6,302.288745,very_high
4560,2252-ISRNH,One year,Fiber optic,9,90.35,767.90,0.544853,6,295.365065,very_high
3302,7774-OJSXI,One year,Fiber optic,31,103.45,3066.45,0.472721,6,293.418013,very_high
6356,3587-PMCOY,One year,Fiber optic,10,98.90,1064.95,0.487500,6,289.282701,very_high
2785,4016-BJKTZ,Two year,Fiber optic,25,108.90,2809.05,0.215025,12,280.994108,high
6628,9979-RGMZT,One year,Fiber optic,7,94.05,633.45,0.489097,6,275.997583,very_high
3287,5828-AVIPD,One year,Fiber optic,19,100.95,1875.55,0.449317,6,272.151533,high
2024,8272-ONJLV,One year,Fiber optic,12,95.70,1184.00,0.469835,6,269.779351,very_high


In [2]:
out_path = "../data/processed/powerbi_scoring_mart.csv"
out.to_csv(out_path, index=False)
out_path


'../data/processed/powerbi_scoring_mart.csv'

In [3]:
print("Total expected revenue loss all customers =", out["expected_revenue_loss"].sum())
print("Very high segment customers =", (out["risk_segment"] == "very_high").sum())


Total expected revenue loss all customers = 497341.6267114965
Very high segment customers = 1761
